# A100 lane job-2 — vision chart embeddings (Stage 1c)

Registered: `hypotheses.jsonl` → `vision-embedding-advisory` (2026-10-23).
Frozen DINOv2-small embeddings of the 3,000 vision-manifest charts
(1,500 champion entries + 1,500 matched randoms, Jay's template).
NO training, NO evaluation here — the judge (sparse3+emb vs sparse3 alone,
time-blocked CV, fit≤2024-12 frozen → 2025+ replication) runs back home.

Rules: no secrets, market data only. Outputs to Drive (one auth click).
Chunked checkpoints: safe to re-run — finished chunks are skipped.
**The final cell unassigns the runtime** so the A100 stops burning credits
the moment artifacts are safely in Drive (do not interrupt mid-run).

Colab gotchas baked in from job-1: pip name is `huggingface_hub`;
pandas 3 removed `Series.view`; Colab's GitHub notebook cache is sticky —
if you edit this notebook, RENAME the file before reopening from GitHub.

In [ ]:
import os
from pathlib import Path

# Pack from the public GitHub release; if you already uploaded pack_job2.zip to
# Drive/brain_packs/, the setup cell prefers that copy (saves the download).
PACK_URL = os.environ.get(
    "BRAIN_PACK_URL",
    "https://github.com/vicjayjay/brain-packs/releases/download/job2-v1/pack_job2.zip")
DRIVE_PACK = "/content/drive/MyDrive/brain_packs/pack_job2.zip"
OUT_DIR = os.environ.get("BRAIN_OUT_DIR", "/content/drive/MyDrive/brain_packs/out_job2")
CHUNK = 256          # charts per checkpoint chunk
BATCH = 64           # images per forward pass on A100
MODEL = "dinov2_vits14"   # frozen; 384-d output
print(PACK_URL)

In [ ]:
import subprocess, sys, zipfile, urllib.request
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "pandas", "pyarrow", "pillow"], check=True)
from google.colab import drive  # type: ignore
drive.mount("/content/drive")   # one consent

work = Path("/content/job2"); work.mkdir(exist_ok=True)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

zip_local = Path(DRIVE_PACK) if Path(DRIVE_PACK).exists() else None
if zip_local is None:
    zip_local = work / "pack_job2.zip"
    if not zip_local.exists():
        urllib.request.urlretrieve(PACK_URL, zip_local)
with zipfile.ZipFile(zip_local) as z:
    z.extractall(work / "pack")
charts = sorted((work / "pack" / "charts_512").glob("*.jpg"))
assert len(charts) == 3000, f"expected 3000 charts, got {len(charts)}"
print(f"pack ready: {len(charts)} charts")

In [ ]:
import torch
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("facebookresearch/dinov2", MODEL).to(device).eval()
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def load_batch(paths):
    return torch.stack([tf(Image.open(p).convert("RGB")) for p in paths]).to(device)

print("model ready on", device, "| emb dim 384")

In [ ]:
import time
import numpy as np
import pandas as pd

n_chunks = int(np.ceil(len(charts) / CHUNK))
t0 = time.time()
for ci in range(n_chunks):
    out_f = Path(OUT_DIR) / f"chunk_{ci:04d}_charts.parquet"
    if out_f.exists():
        continue                                    # resume: finished chunks skip
    part = charts[ci * CHUNK:(ci + 1) * CHUNK]
    ids, embs = [], []
    for bi in range(0, len(part), BATCH):
        bp = part[bi:bi + BATCH]
        with torch.no_grad():
            e = model(load_batch(bp)).float().cpu().numpy()
        ids += [p.stem for p in bp]
        embs.append(e)
    embs = np.concatenate(embs)
    df = pd.DataFrame(embs.astype(np.float16),
                      columns=[f"v{i}" for i in range(embs.shape[1])])
    df.insert(0, "sample_id", ids)
    df.to_parquet(out_f, index=False)               # straight to Drive = checkpoint
    dt = time.time() - t0
    print(f"chunk {ci + 1}/{n_chunks}  {dt:.0f}s elapsed", flush=True)
print("all chunks done")

In [ ]:
import hashlib, json

chunk_files = sorted(Path(OUT_DIR).glob("chunk_*_charts.parquet"))
full = pd.concat([pd.read_parquet(c) for c in chunk_files], ignore_index=True)
assert len(full) == 3000 and full["sample_id"].nunique() == 3000
f = Path(OUT_DIR) / "emb_charts.parquet"
full.to_parquet(f, index=False)
sha = hashlib.sha256(f.read_bytes()).hexdigest()
man = {"job": "vision_embed_job2", "model": MODEL, "rows": int(len(full)),
       "emb_dim": 384, "dtype": "float16", "file": f.name, "sha256": sha}
(Path(OUT_DIR) / "manifest_job2.json").write_text(json.dumps(man, indent=2))
print("emb_charts.parquet", full.shape, "sha256", sha)
print("DONE — artifacts in", OUT_DIR)

In [ ]:
# Resource saving (Jay 10-23): everything durable is already in Drive — the
# parquet + manifest above are the only artifacts. Unassign the runtime so the
# A100 stops billing immediately instead of idling to the timeout.
# NOTE: this DELETES the VM (local /content scratch goes away; Drive persists).
from google.colab import runtime  # type: ignore
runtime.unassign()